<a href="https://colab.research.google.com/github/swetasm108-bit/Grammar-Scoring-Engine/blob/main/Grammar_Scoring_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q resampy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.9 MB/s eta 0:00:00


In [2]:
# Clone the project from GitHub
!git clone https://github.com/sm6746/SHL.git

# Move into the project directory
%cd SHL

Cloning into 'SHL'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 22 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 19.12 KiB | 3.19 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/kaggle/working/SHL


In [3]:
!pip install openai-whisper gingerit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=ba82e5691698d86a5c824e3ac8b7fc3c81a4eebcc5605fd4c21c9bdef380b323
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
  Created wheel for gingerit: filename=gingerit-0.0.0.1-py3-none-any.whl size=1305 sha256=91e623d7db6c096d1dc914f8a9d1a566bbda0c418fc20e2cafc3bbb549fc91d4
  Stored in directory: /root/.cache/pip/wheels/23/a7/02/2ea6493213411d9a392886e97d43684febd4fbd3d5519af183
Successfully built openai-whisper gingerit


In [4]:
# Download the shl-sarah-project dataset
!kaggle datasets download -d sarahkhambatta/shl-sarah-project

# Unzip the files into a folder named 'shl_data'
!mkdir -p shl_data
!unzip -q shl-sarah-project.zip -d shl_data

# List files to verify
!ls shl_data

Dataset URL: https://www.kaggle.com/datasets/sarahkhambatta/shl-sarah-project
License(s): unknown
100%|██████████████████████████████████████| 6.52k/6.52k [00:00<00:00, 19.1MB/s]

app.py		README.md	  sample_submission.csv  train.csv
notebook.ipynb	requirements.txt  test.csv


In [5]:
import pandas as pd

# Update these paths to match your sidebar exactly
train_df = pd.read_csv('/kaggle/input/datasets/swetamishraai/dataset/train.csv')
test_df = pd.read_csv('/kaggle/input/datasets/swetamishraai/dataset/test.csv')

# Verify it worked
print("Train Data Loaded:", train_df.shape)
print("Test Data Loaded:", test_df.shape)

Train Data Loaded: (444, 2)
Test Data Loaded: (195, 1)


In [6]:
import pandas as pd
import numpy as np
import librosa
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


np.random.seed(42)
tf.random.set_seed(42)

2026-03-01 18:16:40.994898: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772389001.391227      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772389001.489028      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772389002.428741      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772389002.428800      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772389002.428803      23 computation_placer.cc:177] computation placer alr

In [7]:
def extract_audio_features(file_path):
    try:
        # Load audio file
        audio, sample_rate = librosa.load(file_path, res_type='kaiser_fast')

        # Extract MFCCs (standard is 40 features)
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
        mfccs_scaled = np.mean(mfccs.T, axis=0)

        return mfccs_scaled
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [8]:
import pandas as pd
import os

# Precise Kaggle paths from your sidebar
train_csv_path = '/kaggle/input/datasets/swetamishraai/dataset/train.csv'
test_csv_path = '/kaggle/input/datasets/swetamishraai/dataset/test.csv'

# Check if files exist before reading
if os.path.exists(train_csv_path) and os.path.exists(test_csv_path):
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)
    print("✅ Files found! Test Data Head:")
    print(test_df.head())
else:
    print("❌ Files still not found. Please click the 'Copy File Path' icon next to test.csv in your sidebar and paste it here.")

✅ Files found! Test Data Head:
         filename
0   audio_706.wav
1   audio_800.wav
2    audio_68.wav
3  audio_1267.wav
4   audio_683.wav


In [9]:
from tqdm.notebook import tqdm
import os

In [10]:
from tqdm.notebook import tqdm
import os

# Ensure this matches your sidebar exactly
TEST_AUDIO_DIR = '/kaggle/input/datasets/swetamishraai/audio-test'

test_features = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_path = os.path.join(TEST_AUDIO_DIR, row['filename'])

    if os.path.exists(audio_path):
        try:
            # Ensure this function cell was run previously!
            features = extract_audio_features(audio_path)
            if features is not None:
                test_features.append(features)
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
    else:
        if i < 5:
            print(f"❌ Still missing: {audio_path}")

  0%|          | 0/195 [00:00<?, ?it/s]

In [11]:
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

# Ensure your paths are correct
TEST_AUDIO_DIR = '/kaggle/input/datasets/swetamishraai/audio-test'
test_features = []

# Re-run the loop
for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_path = os.path.join(TEST_AUDIO_DIR, row['filename'])

    if os.path.exists(audio_path):
        try:
            # The error happened here, but resampy is now installed
            audio, sr = librosa.load(audio_path, sr=22050)

            # Extract features (e.g., MFCCs)
            mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
            mfccs_scaled = np.mean(mfccs.T, axis=0)
            test_features.append(mfccs_scaled)

        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
    else:
        if i < 3: print(f"File missing: {audio_path}")

print(f"\n✅ Success! Processed {len(test_features)} test samples.")

100%|██████████| 195/195 [00:26<00:00,  7.39it/s]


✅ Success! Processed 195 test samples.


In [12]:
# Assuming you found your training audio folder earlier
TRAIN_AUDIO_DIR = '/kaggle/input/datasets/swetamishraai/audio-train'

train_features = []
train_labels = []

print("Extracting features from training set...")
for i, row in tqdm(train_df.iterrows(), total=len(train_df)):
    audio_path = os.path.join(TRAIN_AUDIO_DIR, row['filename'])
    if os.path.exists(audio_path):
        try:
            audio, sr = librosa.load(audio_path, sr=22050)
            mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
            train_features.append(np.mean(mfccs.T, axis=0))
            train_labels.append(row['label'])
        except Exception as e:
            continue

# Convert to arrays
X_train = np.array(train_features)
y_train = np.array(train_labels)
X_test = np.array(test_features) # This is the list from your last image

Extracting features from training set...


100%|██████████| 444/444 [01:16<00:00,  5.77it/s]


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

# We name it 'scoring_model' so it doesn't overwrite your 'model' (Whisper)
scoring_model = Sequential([
    Dense(256, activation='relu', input_shape=(40,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1)
])

scoring_model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
print("Scoring model initialized!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1772389258.706041      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772389258.711937      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Scoring model initialized!


In [14]:
# Train the NEW scoring_model for 20 epochs
history = scoring_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20


I0000 00:00:1772389262.211601     123 service.cc:152] XLA service 0x7e9c9400c130 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772389262.211638     123 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1772389262.211642     123 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1772389262.673775     123 cuda_dnn.cc:529] Loaded cuDNN version 91002


 1/12 ━━━━━━━━━━━━━━━━━━━━ 44s 4s/step - loss: 8.5697 - mae: 2.7012

I0000 00:00:1772389264.748688     123 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


12/12 ━━━━━━━━━━━━━━━━━━━━ 6s 212ms/step - loss: 6.0445 - mae: 2.1385 - val_loss: 866.5706 - val_mae: 29.1297
Epoch 2/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3.2832 - mae: 1.3920 - val_loss: 250.8822 - val_mae: 15.5705
Epoch 3/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.7618 - mae: 1.0368 - val_loss: 102.5939 - val_mae: 9.9087
Epoch 4/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.4436 - mae: 0.9411 - val_loss: 24.9097 - val_mae: 4.8191
Epoch 5/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.4641 - mae: 0.9751 - val_loss: 16.5631 - val_mae: 3.8982
Epoch 6/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.2759 - mae: 0.9133 - val_loss: 5.3459 - val_mae: 2.1290
Epoch 7/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.3434 - mae: 0.9508 - val_loss: 1.9214 - val_mae: 1.1694
Epoch 8/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.1172 - mae: 0.8516 - val_loss: 0.5580 - val_mae: 0.6021
Epoch 9/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.0895 - mae:

In [15]:
# Use the trained Keras model to predict scores for the test features
# We flatten the results so they fit into a simple column
predictions = scoring_model.predict(X_test).flatten()

print(f"✅ Generated {len(predictions)} predictions.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
✅ Generated 195 predictions.


In [16]:
# Create the final DataFrame
submission = pd.DataFrame({
    'filename': test_df['filename'].iloc[:195],
    'label': predictions
})

# Optional: Ensure scores stay within the standard 0-5 range
submission['label'] = submission['label'].clip(0, 5)

# Save to the Colab environment
submission.to_csv('submission.csv', index=False)

print("🚀 submission.csv is ready!")

🚀 submission.csv is ready!
